# In-Context Learning


In-context learning is a generalisation of few-shot learning where the LLM is provided a context as part of the prompt and asked to respond by utilising the information in the context.

* Example: *"Summarize this research article into one paragraph highlighting its strengths and weaknesses: [insert article text]”*
* Example: *"Extract all the quotes from this text and organize them in alphabetical order: [insert text]”*

A very popular technique that you will learn in week 5 called Retrieval-Augmented Generation (RAG) is a form of in-context learning, where:
* a search engine is used to retrieve some relevant information
* that information is then provided to the LLM as context


In this example we download some recent research papers from arXiv papers, extract the text from the PDF files and ask Gemini to summarize the articles as well as provide the main strengths and weaknesses of the papers. Finally we print the summaries to a local html file and as markdown.

In [38]:
!pip install requests bs4 google-generativeai pypdf

799.79s - pydevd: Sending message related to process being replaced timed-out after 5 seconds


In [39]:
import os
import requests
from bs4 import BeautifulSoup
import google.generativeai as genai
from urllib.request import urlopen, urlretrieve
from IPython.display import Markdown, display
from pypdf import PdfReader
from datetime import date
from tqdm import tqdm
# from google.colab import userdata

In [ ]:
API_KEY = ""
# API_KEY = userdata.get('GEMINI_API_KEY')
genai.configure(api_key=API_KEY)

We select those papers that have been featured in Hugging Face papers.

In [41]:
BASE_URL = "https://huggingface.co/papers"
page = requests.get(BASE_URL)
soup = BeautifulSoup(page.content, "html.parser")
h3s = soup.find_all("h3")[:5]

papers = []

for h3 in h3s:
    a = h3.find("a")
    title = a.text
    link = a["href"].replace('/papers', '')

    papers.append({"title": title, "url": f"https://arxiv.org/pdf{link}"})

Code to extract text from PDFs.

In [42]:
def extract_paper(url):
    html = urlopen(url).read()
    soup = BeautifulSoup(html, features="html.parser")

    # kill all script and style elements
    for script in soup(["script", "style"]):
        script.extract()    # rip it out

    # get text
    text = soup.get_text()

    # break into lines and remove leading and trailing space on each
    lines = (line.strip() for line in text.splitlines())
    # break multi-headlines into a line each
    chunks = (phrase.strip() for line in lines for phrase in line.split("  "))
    # drop blank lines
    text = '\n'.join(chunk for chunk in chunks if chunk)

    return text


def extract_pdf(url):
    import tempfile
    import os
    
    # Create a temporary file
    temp_file = tempfile.NamedTemporaryFile(delete=False, suffix='.pdf')
    temp_filename = temp_file.name
    temp_file.close()
    
    try:
        # Download PDF to temporary file using requests (handles SSL better)
        response = requests.get(url, verify=True, timeout=30)
        response.raise_for_status()
        
        with open(temp_filename, 'wb') as f:
            f.write(response.content)
        
        reader = PdfReader(temp_filename)
        text = ""
        for page in reader.pages:
            text += page.extract_text() + "\n"
        return text
    finally:
        # Clean up temporary file
        if os.path.exists(temp_filename):
            os.remove(temp_filename)


def printmd(string):
    display(Markdown(string))

In [43]:
LLM = "gemini-2.5-flash"
model = genai.GenerativeModel(LLM)

We use Gemini to summarize the papers.

In [44]:
for paper in tqdm(papers):
    try:
        pdf_text = extract_pdf(paper["url"])
        message = (
            "Summarize this research article and present the strengths and weaknesses in a markdown table format. "
            "The table should have two columns: 'Strengths' and 'Weaknesses'. "
            "Include a brief summary paragraph before the table. "
            "Format the output as:\n\n"
            "Summary: [brief summary paragraph]\n\n"
            "| Strengths | Weaknesses |\n"
            "|-----------|------------|\n"
            "| [strength 1] | [weakness 1] |\n"
            "| [strength 2] | [weakness 2] |\n"
            "| ... | ... |\n\n"
            + pdf_text
        )
        response = model.generate_content(message)
        paper["summary"] = response.text
    except Exception as e:
        print(f"Generation failed for {paper['title']}: {type(e).__name__}: {str(e)}")
        paper["summary"] = "Paper not available"

100%|██████████| 5/5 [01:07<00:00, 13.51s/it]


We print the results to a html file.

In [45]:
import re

def markdown_to_html_table(markdown_text):
    """Convert markdown table to HTML table with improved formatting"""
    
    def parse_table(table_text):
        """Parse a markdown table and return HTML"""
        lines = [line.strip() for line in table_text.strip().split('\n') if line.strip()]
        if len(lines) < 2:
            return table_text
        
        # Parse header (first line)
        header_line = lines[0]
        headers = [h.strip() for h in header_line.split('|') if h.strip()]
        
        # Skip separator line (second line with dashes)
        # Parse data rows
        rows = []
        for line in lines[2:]:
            if line.strip().startswith('|'):
                cells = [c.strip() for c in line.split('|') if c.strip() or c == '']
                # Remove empty first/last cells from split
                if cells and not cells[0]:
                    cells = cells[1:]
                if cells and not cells[-1]:
                    cells = cells[:-1]
                if len(cells) >= len(headers):
                    rows.append(cells[:len(headers)])
        
        # Build HTML table
        html = '<table border="1" cellpadding="8" cellspacing="0" style="border-collapse: collapse; width: 100%; margin: 15px 0; font-size: 14px;">\n'
        html += '<thead><tr>\n'
        for h in headers:
            html += f'  <th style="text-align: left; background-color: #4a90e2; color: white; padding: 10px; font-weight: bold;">{h}</th>\n'
        html += '</tr></thead>\n<tbody>\n'
        
        for row in rows:
            html += '<tr>\n'
            for i, cell in enumerate(row):
                # Convert markdown bold to HTML
                cell_html = re.sub(r'\*\*(.+?)\*\*', r'<strong>\1</strong>', cell)
                # Alternate row colors for better readability
                bg_color = '#f9f9f9' if len(rows) > 0 and rows.index(row) % 2 == 0 else '#ffffff'
                html += f'  <td style="padding: 8px; vertical-align: top; background-color: {bg_color};">{cell_html}</td>\n'
            html += '</tr>\n'
        
        html += '</tbody></table>'
        return html
    
    # Pattern to match markdown tables (more flexible)
    # Matches: | header | header | followed by |---| followed by | data | data |
    table_pattern = r'(\|[^\n]+\|\n\|[-\s\|:]+\|\n(?:\|[^\n]+\|\n?)+)'
    
    def replace_table(match):
        return parse_table(match.group(1))
    
    # Replace markdown tables with HTML tables
    result = re.sub(table_pattern, replace_table, markdown_text, flags=re.MULTILINE)
    
    # Convert other markdown to HTML
    # Bold text
    result = re.sub(r'\*\*(.+?)\*\*', r'<strong>\1</strong>', result)
    # Italic text
    result = re.sub(r'\*(.+?)\*', r'<em>\1</em>', result)
    # Paragraphs (double newlines)
    paragraphs = result.split('\n\n')
    html_paragraphs = []
    for para in paragraphs:
        para = para.strip()
        if para and not para.startswith('<table'):  # Don't wrap tables in <p>
            # Convert single newlines to <br> within paragraphs
            para = para.replace('\n', '<br>\n')
            html_paragraphs.append(f'<p style="margin: 10px 0; line-height: 1.6;">{para}</p>')
        else:
            html_paragraphs.append(para)
    
    return '\n'.join(html_paragraphs)

page = f"""<html>
<head>
  <title>Daily Dose of AI Research</title>
  <style>
    body {{ font-family: Arial, sans-serif; margin: 20px; }}
    h1 {{ color: #333; }}
    h2 {{ color: #555; margin-top: 30px; }}
    table {{ border-collapse: collapse; width: 100%; margin: 15px 0; }}
    th, td {{ border: 1px solid #ddd; padding: 8px; text-align: left; }}
    th {{ background-color: #f2f2f2; font-weight: bold; }}
  </style>
</head>
<body>
  <h1>Daily Dose of AI Research</h1>
  <h4>{date.today()}</h4>
  <p><i>Summaries generated with: {LLM}</i></p>
"""
with open("papers.html", "w") as f:
    f.write(page)
for paper in papers:
    html_summary = markdown_to_html_table(paper["summary"])
    page = f'<h2><a href="{paper["url"]}">{paper["title"]}</a></h2>{html_summary}'
    with open("papers.html", "a") as f:
        f.write(page)
end = "</body></html>"
with open("papers.html", "a") as f:
    f.write(end)

We can also print the results to this notebook as markdown.

In [46]:
for paper in papers:
    printmd("**[{}]({})**\n\n{}".format(paper["title"],
                                        paper["url"],
                                        paper["summary"]))

**[HaluMem: Evaluating Hallucinations in Memory Systems of Agents](https://arxiv.org/pdf/2511.03506)**

Summary: This research article introduces HaluMem (Hallucination in Memory Benchmark), the first operation-level evaluation benchmark designed to localize and measure hallucinations in AI agent memory systems. Unlike previous end-to-end question-answering evaluations, HaluMem defines three tasks—memory extraction, memory updating, and memory question answering—to pinpoint where errors like fabrication, omissions, and conflicts arise. The benchmark includes two user-centric, multi-turn human-AI interaction datasets, HaluMem-Medium and HaluMem-Long, featuring approximately 15,000 memory points and 3,500 questions, with contexts extending to over 1 million tokens. Empirical studies using HaluMem reveal that current memory systems frequently generate and accumulate hallucinations during extraction and updating, which then propagate to the question-answering stage, particularly in long-context scenarios. The authors call for future research into more interpretable and constrained memory operation mechanisms to improve reliability.

| Strengths | Weaknesses |
|---|---|
| Addresses a critical gap by providing the first operation-level benchmark for memory hallucination, allowing for detailed error localization rather than just end-to-end performance. | The evaluation process relies on GPT-4o for consistency determination and scoring, which, while efficient, might introduce its own biases or limitations from the evaluating LLM. |
| Features a comprehensive evaluation framework with three distinct operational tasks (memory extraction, updating, and question answering) for a thorough assessment of memory systems. | One of the evaluated systems (Zep) could not be fully benchmarked across all tasks due to API limitations, resulting in incomplete comparative analysis for that system. |
| Constructs extensive, high-quality, user-centric multi-turn human-AI interaction datasets (HaluMem-Medium and HaluMem-Long) with dynamic personas and adversarial content. | The memory updating evaluation's high omission rates are significantly influenced by preceding memory extraction failures, suggesting that the update task's assessment may be heavily dependent on upstream performance rather than isolating update-specific issues. |
| Includes HaluMem-Long, specifically designed to test memory systems under ultra-long context conditions (over 1M tokens), evaluating robustness against irrelevant information and context scaling challenges. | While adversarial content and irrelevant dialogues are inserted for long-context evaluation, the specific nature (e.g., factual Q&A, mathematical reasoning) might not fully represent the diversity of conversational noise or topic drift in real-world human-AI interactions. |
| The six-stage dataset construction pipeline, coupled with human annotation for quality assurance, ensures the coherence, consistency, and reliability of the benchmark data. | Some state-of-the-art memory systems exhibited very high computational costs (dialogue addition time) during evaluation, highlighting efficiency challenges that could limit practical deployment, though this is a finding *from* the benchmark rather than a direct weakness *of* the benchmark itself. |
| Empirical studies using HaluMem effectively reveal how hallucinations accumulate and propagate from earlier memory stages (extraction, updating) to impact later stages (question answering), offering valuable insights for mitigation strategies. | The definition of "hallucination" in the benchmark primarily focuses on factual inaccuracies, inconsistencies, and omissions, potentially overlooking other subtle forms of hallucination (e.g., stylistic drift, misrepresentation of sentiment) that could occur in memory systems. |

**[IterResearch: Rethinking Long-Horizon Agents via Markovian State
  Reconstruction](https://arxiv.org/pdf/2511.07327)**

Summary: This research article introduces IterResearch, a novel iterative deep-research paradigm designed to overcome the limitations of existing "mono-contextual" LLM agents on long-horizon tasks, specifically context suffocation and noise contamination. IterResearch reformulates long-horizon research as a Markov Decision Process, employing a strategic workspace reconstruction mechanism that maintains an evolving report as memory and periodically synthesizes insights, thereby preserving consistent reasoning capacity. It also introduces Efficiency-Aware Policy Optimization (EAPO), a reinforcement learning framework that incentivizes efficient exploration through geometric reward discounting and stable distributed training. The authors demonstrate that IterResearch significantly outperforms open-source agents, narrows the gap with proprietary systems, enables unprecedented interaction scaling up to 2048 turns, facilitates cross-paradigm knowledge transfer, and serves as an effective model-agnostic prompting strategy for frontier LLMs.

| Strengths | Weaknesses |
|---|---|
| **Novel Iterative Paradigm:** Addresses fundamental limitations of mono-contextual agents (context suffocation, noise contamination) with a principled Markovian workspace reconstruction. | **Reliance on LLM Synthesis Quality:** The effectiveness of the "evolving report" heavily depends on the LLM's ability to accurately and comprehensively synthesize information and filter noise, which could be a point of failure if synthesis is imperfect. |
| **Sustained Reasoning Capacity:** Maintains consistent reasoning quality and a bounded memory footprint across arbitrary exploration depths, enabling engagement with significantly longer tasks. | **Complex Two-Stage Training:** Involves both rejection sampling fine-tuning (RFT) and reinforcement learning (RL) with specific "learning zone selection," indicating a non-trivial and potentially resource-intensive training process. |
| **Significant Performance Gains:** Achieves an average +14.5pp improvement over state-of-the-art open-source agents and narrows the performance gap with frontier proprietary systems across diverse benchmarks. | **Hyperparameter Sensitivity (EAPO):** The geometric reward discounting in EAPO introduces a `gamma` hyperparameter, whose optimal value might vary across different task types or domains, potentially requiring careful tuning. |
| **Unprecedented Interaction Scaling:** Demonstrates the ability to scale to 2048 interactions within a constant context window, a feat structurally infeasible for most existing approaches, proving its capability for extremely long-horizon tasks. | **Practical Resource Consumption for Extreme Depths:** While theoretically unbounded, operating at 2048 interactions still entails a very large number of LLM inferences and tool calls, leading to substantial computational and API costs in real-world deployment. |
| **Efficiency-Aware Policy Optimization (EAPO):** Incentivizes efficient exploration through geometric reward discounting, leading to shorter average trajectories while maintaining or improving accuracy. | **"Narrows the gap" not "closes the gap":** Despite significant improvements, the paper indicates that there is still room for IterResearch to fully surpass all frontier proprietary systems on every benchmark. |
| **Effective Model-Agnostic Prompting Strategy:** Provides substantial performance improvements for frontier models (like o3 and DeepSeek-V3.1) over ReAct, even without additional training, highlighting its broad applicability. | **Evaluation primarily relies on accuracy:** For complex research tasks, other aspects like the coherence, novelty, or depth of insights in the final report, beyond mere correctness, could also be important but are harder to quantify and are not explicitly measured. |
| **Cross-Paradigm Knowledge Transfer:** The trajectories generated by IterResearch can enhance mono-contextual agents, suggesting it induces superior exploration behaviors and produces high-quality training signals. | |
| **Consistent Computational Efficiency:** Maintains constant computational cost per round (O((|M|+|TR|) 2)), independent of trajectory length, unlike quadratically increasing costs in mono-contextual approaches. | |

**[DRIVE: Data Curation Best Practices for Reinforcement Learning with
  Verifiable Reward in Competitive Code Generation](https://arxiv.org/pdf/2511.06307)**

Summary: This research article introduces DRIVE, a two-stage reinforcement learning with verifiable reward (RLVR) framework designed for competitive programming code generation, an underexplored area where data curation receives less attention than algorithmic design. The pipeline begins with supervised fine-tuning (SFT) from strong open-source models. The subsequent RL process first focuses on "Entropy Expansion" using a large, uniformly distributed problem set with moderate rollouts to increase output diversity and reduce repetitive generation. The second stage, "Hard-Focus Curriculum" (Pre-GRPO), targets challenging problems with a high rollout budget, continuously retaining difficult instances to push the model's capabilities. Implemented on Qwen2.5-32B, the method achieves state-of-the-art performance among models of similar scale and performs comparably to much larger leading systems on recent LeetCode and Codeforces contests, demonstrating significant improvements over the SFT baseline and other 32B models. The study also validates positive scaling trends on an internal large-scale Mixture-of-Experts (MoE) model.

| Strengths | Weaknesses |
|---|---|
| Addresses an underexplored domain: RLVR specifically for competitive programming code generation. | Computational cost: The method, particularly the second RL stage with large rollout budgets (64 rollouts), requires substantial computational resources, which may be a barrier for wider adoption. |
| Novel two-stage RL framework: "Entropy Expansion" followed by "Hard-Focus Curriculum" effectively tackles limitations like low entropy, repetitive generation, and difficulty with challenging problems. | Dependence on "hard" problem identification: The effectiveness relies on accurately identifying and curating "hard" problems, which might be subjective or dataset-specific, and the initial classification model's details are not extensively elaborated. |
| Strong empirical validation: Extensive evaluation on recent, uncontaminated LeetCode and Codeforces weekly contests ensures avoidance of data leakage and robust generalization assessment. | Limited open access: The paper mentions an "internal large-scale MoE model" for scaling experiments, but details on its architecture, parameters, and training specifics are not provided, limiting reproducibility and transparency of scaling results. |
| Achieves state-of-the-art performance: The 32B parameter model outperforms other models of similar scale and achieves performance comparable to much larger state-of-the-art systems (e.g., DeepSeek v3.1), demonstrating impressive efficiency. | Lack of deeper algorithmic contribution: While effectively utilizing GRPO, the paper's primary focus is on data curation and curriculum design, with less emphasis on novel RL algorithm development or comparative analysis of different RL algorithms. |
| Comprehensive ablation studies: Detailed ablation studies validate the necessity and effectiveness of each stage of the RL framework and the importance of difficulty-aware training strategies. | Partial training data overlap in evaluation: LiveCodeV6, used as part of the challenging problem set for the second RL training stage, is also included in the evaluation benchmarks. While other benchmarks are uncontaminated, this could potentially inflate reported performance on LiveCodeV6 as it was part of the training distribution. |
| Positive scaling trends: Demonstrates that the proposed strategies are effective and scale well to larger models (internal MoE model), suggesting broader applicability. | The paper does not mention releasing the code, datasets, or trained models, which limits reproducibility and the ability of other researchers to build upon this work directly. |

**[The Station: An Open-World Environment for AI-Driven Discovery](https://arxiv.org/pdf/2511.06309)**

Summary: The research introduces "The Station," an open-world, multi-agent environment designed to facilitate autonomous scientific discovery by AI agents. Unlike traditional rigid, centralized pipelines, the Station allows agents to engage in long, unscripted scientific journeys, including reading papers, formulating hypotheses, writing code, conducting analyses, and publishing results. Experiments demonstrate that AI agents within the Station achieve new state-of-the-art performance across diverse benchmarks (mathematics, computational biology, machine learning), discovering novel methods such as a density-adaptive algorithm for scRNA-seq batch integration and a Fourier-based architecture for neural activity prediction. The environment fosters emergent narratives, collaboration, and knowledge accumulation across agent generations. An "Open Station" variant, without explicit goals, also revealed insights into agents forming societies and even collective delusions when external objective signals are absent. The work advocates for rich, open environments to unlock AI's full potential in scientific research, moving beyond mere optimization.

| Strengths | Weaknesses |
|---|---|
| **Novel Open-World Paradigm:** Introduces a unique multi-agent, unscripted environment for autonomous scientific discovery, fostering complex, iterative research beyond rigid pipelines. | **High Computational & Cost Requirements:** Running the environment with advanced LLMs (Gemini, GPT-5, Claude) incurs substantial API costs and requires significant computational resources. |
| **Achieves State-of-the-Art Results:** Demonstrates superior performance on diverse, challenging benchmarks across mathematics, computational biology, and machine learning, setting new SOTA records. | **Long Simulation Runtimes:** Scientific journeys can span thousands of "Station Ticks" and weeks of wall-clock time, hindering rapid experimentation and iteration. |
| **Fosters Discovery of Novel Methods:** Agents develop genuinely original algorithms and apply out-of-domain concepts (e.g., density-awareness from clustering to batch integration), indicating creative problem-solving. | **High Variance in Outcomes:** The paper notes that "Station instances exhibit high variance," implying that achieving optimal results may require numerous, costly, and time-consuming runs. |
| **Enables Emergent Scientific Narratives:** Facilitates complex, unscripted research journeys, collaboration, reflection, and "publication," mimicking the messy and non-linear processes of human scientific breakthroughs. | **Susceptibility to Stagnation:** The need for an explicit "Stagnation Protocol" suggests agents can get stuck in local optima, indicating a limitation in purely emergent, unguided exploration. |
| **Accumulation of Persistent Knowledge:** Allows agents to build upon a cumulative history through shared archives, public forums, and private lineage records, enabling long-term research continuity. | **"Open Station" Delusion Risk:** Without explicit objectives or external signals, agents developed collective delusions detached from reality, highlighting challenges in truly open-ended, unguided discovery. |
| **High Agent Autonomy:** Provides agents significant freedom in choosing actions, rooms to visit, and developing their own research paths, fostering creativity and open-ended exploration. | **Dependence on Advanced LLMs:** Performance is inherently tied to the capabilities and high costs of the latest, most powerful large language models, limiting accessibility. |
| **Robust Auxiliary Systems:** Includes a reviewer system for paper quality, a debugger for code errors, a maturity system for structured growth, and a stagnation protocol to encourage new avenues. | **Evaluation Speed Constraints:** The requirement for "Fast Evaluation" (e.g., within 2 hours wall-clock time) might limit the types of scientific tasks that can be effectively tackled within the environment. |
| **Transparent and Reproducible:** Source code and full agent dialogue data are publicly available, promoting community engagement, reproducibility, and further research. | **"Novelty" Nuance:** While solutions are described as novel, some are framed as "unified designs" or "first applications" of existing concepts, rather than entirely new fundamental ideas. |

**[MVU-Eval: Towards Multi-Video Understanding Evaluation for Multimodal
  LLMs](https://arxiv.org/pdf/2511.07250)**

Summary: This research article introduces MVU-Eval, the first comprehensive benchmark designed to evaluate Multimodal Large Language Models (MLLMs) on multi-video understanding tasks. Addressing a significant gap in existing benchmarks that primarily focus on single-video analysis, MVU-Eval comprises 1,824 meticulously curated question-answer pairs spanning 4,959 videos from diverse real-world domains. It assesses eight core competencies, including fundamental perception tasks (Object Recognition, Spatial Understanding, Counting, Comparison) and high-order reasoning tasks (Knowledge-Intensive Reasoning, In-Context Learning, Retrieval-Augmented Generation, Temporal Reasoning). Extensive evaluations of state-of-the-art MLLMs reveal significant performance discrepancies and limitations, highlighting the challenges MLLMs face in effectively understanding and reasoning across multiple video inputs. The benchmark will be made publicly available to foster future research in this critical area.

| Strengths | Weaknesses |
|---|---|
| Addresses a critical and unaddressed gap by being the first comprehensive benchmark for multi-video understanding, crucial for real-world MLLM applications. | Video samples are relatively short in duration, limiting the assessment of long-range temporal reasoning and narrative comprehension over extended sequences (e.g., movie-level lengths). |
| Covers a broad range of capabilities, evaluating eight core competencies that span both fundamental multi-video perception and high-order reasoning tasks. | The benchmark currently focuses solely on visual inputs and does not support the evaluation of models on audio, omitting a crucial modality for comprehensive real-world understanding. |
| Features a large and diverse dataset with 1,824 QA pairs across 4,959 videos from various domains (e.g., autonomous driving, sports, gaming, movies), ensuring robust evaluation. | Some MLLMs struggle with instruction-following, generating free-form text instead of specific answer options, which can complicate automated evaluation and parsing of results. |
| Employs a rigorous data collection and quality control pipeline, including automated generation, extensive human verification, and systematic leakage removal to ensure high-quality and challenging QA pairs. | Despite models specialized for long-video understanding being included, their performance remains modest, indicating that effective inter-video fusion and alignment mechanisms are still significant, unresolved challenges for MLLMs. |
| Reveals significant limitations and performance discrepancies in current state-of-the-art MLLMs (both open and closed-source), demonstrating that the benchmark is challenging and identifies clear areas for improvement. | |
| Provides insightful findings on factors influencing MLLM performance, such as model scaling, number of input frames, input resolution, and input format, offering clear directions for future research. | |
| The benchmark is publicly available, fostering collaborative research and development in the field of multi-video understanding for MLLMs. | |